# X Full-Archive Keyword Scraper

Notebook-only scraper for `GET /2/tweets/search/all`.

References:
- https://docs.x.com/x-api/posts/search-all-posts
- https://docs.x.com/x-api/posts/search/integrate/operators
- https://docs.x.com/x-api/posts/search/quickstart/full-archive-search

Set `X_BEARER_TOKEN` in your environment before running the scrape cells. The notebook saves a flattened CSV, trimmed JSONL API pages, and a manifest under `data/raw/twitter/`.

In [14]:
from __future__ import annotations

import csv
import json
import os
import re
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
import requests


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "docs" / "keywords").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing docs/keywords and data directories.")


ROOT_DIR = find_repo_root()
KEYWORD_ROOT_DIR = ROOT_DIR / "docs" / "keywords"
KEYWORD_DIR = KEYWORD_ROOT_DIR / "by_language"
OUTPUT_DIR = ROOT_DIR / "data" / "raw" / "twitter"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root : {ROOT_DIR}")
print(f"Keywords  : {KEYWORD_DIR}")
print(f"Output dir: {OUTPUT_DIR}")

Repo root : /Users/angelodelapaz/Documents/GitHub/HealthPH-Plus
Keywords  : /Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/docs/keywords/by_language
Output dir: /Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/data/raw/twitter


## Configuration

`START_TIME` and `END_TIME` must use UTC ISO format: `YYYY-MM-DDTHH:mm:ssZ`. Leave either as `None` for a broad/default API window. For a bounded scrape, set values like `START_TIME = "2026-01-01T00:00:00Z"`.

In [15]:
# Required auth. Do not hard-code tokens in the notebook.
BEARER_TOKEN = os.getenv("X_BEARER_TOKEN")

# Full-archive search settings. Leave START_TIME/END_TIME as None unless intentionally bounding the scrape.
START_TIME = None
END_TIME = None
MAX_RESULTS = 100
SORT_ORDER = "recency"

# Keep requests public-safe and minimal. `public_metrics` is needed for `like_count`.
DEFAULT_FIELD_PROFILE = "public"

# Keep this small for a test run, then raise it when the output looks correct.
MAX_PAGES_PER_QUERY = 3
REQUEST_SLEEP_SECONDS = 1.0

# Self-serve full-archive search query limit. Raise to 4096 only if your account supports it.
QUERY_CHAR_LIMIT = 1024

# Optional filters appended to every query. Examples: ["-is:retweet"], ["place_country:PH"].
EXTRA_QUERY_FILTERS: list[str] = []

# Load language keyword CSVs only, including nested language folders if present.
KEYWORD_FILES = sorted({path.resolve() for path in KEYWORD_DIR.rglob("*.csv")})
KEYWORD_COLUMN_BY_FILE: dict[str, str] = {}

RUN_ID = datetime.now(timezone.utc).strftime("x_full_archive_%Y%m%dT%H%M%SZ")
CSV_OUTPUT_FILE = OUTPUT_DIR / f"{RUN_ID}.csv"
JSONL_OUTPUT_FILE = OUTPUT_DIR / f"{RUN_ID}.jsonl"
MANIFEST_OUTPUT_FILE = OUTPUT_DIR / f"{RUN_ID}_manifest.json"

print(f"Run ID        : {RUN_ID}")
print(f"CSV           : {CSV_OUTPUT_FILE}")
print(f"JSONL         : {JSONL_OUTPUT_FILE}")
print(f"Manifest      : {MANIFEST_OUTPUT_FILE}")
print(f"Token set     : {bool(BEARER_TOKEN)}")
print(f"Field profile : {DEFAULT_FIELD_PROFILE}")

Run ID        : x_full_archive_20260515T062632Z
CSV           : /Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/data/raw/twitter/x_full_archive_20260515T062632Z.csv
JSONL         : /Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/data/raw/twitter/x_full_archive_20260515T062632Z.jsonl
Manifest      : /Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/data/raw/twitter/x_full_archive_20260515T062632Z_manifest.json
Token set     : True
Field profile : public


In [16]:
def normalize_keyword(value: Any) -> str | None:
    if value is None or pd.isna(value):
        return None
    keyword = str(value).strip().strip('"').strip("'")
    keyword = re.sub(r"\s+", " ", keyword)
    return keyword or None


def split_keyword_cell(value: Any) -> list[str]:
    keyword = normalize_keyword(value)
    if not keyword:
        return []
    try:
        return [part.strip() for part in next(csv.reader([keyword])) if part.strip()]
    except csv.Error:
        return [part.strip() for part in keyword.split(",") if part.strip()]


def load_first_column_keywords(path: Path) -> list[str]:
    keywords: list[str] = []
    header_names = {"keyword", "keywords", "term", "terms", "symptom", "symptoms"}
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        for row_number, row in enumerate(csv.reader(f)):
            if not row:
                continue
            first_cell = normalize_keyword(row[0])
            if row_number == 0 and first_cell and first_cell.casefold() in header_names:
                continue
            keywords.extend(split_keyword_cell(row[0]))
    return keywords


def load_keywords_from_file(path: Path, column: str | None = None) -> list[str]:
    if column is None:
        return load_first_column_keywords(path)

    df = pd.read_csv(path, dtype=str).fillna("")
    if df.empty:
        return []
    if column not in df.columns:
        raise ValueError(f"Column '{column}' not found in {path}. Found: {df.columns.tolist()}")

    keywords: list[str] = []
    for value in df[column].tolist():
        keywords.extend(split_keyword_cell(value))
    return keywords


def load_keywords_from_files(paths: list[Path]) -> list[str]:
    seen: set[str] = set()
    merged: list[str] = []
    for path in paths:
        column = KEYWORD_COLUMN_BY_FILE.get(path.name)
        for keyword in load_keywords_from_file(path, column=column):
            key = keyword.casefold()
            if key in seen:
                continue
            seen.add(key)
            merged.append(keyword)
    return merged


def quote_query_term(keyword: str) -> str:
    keyword = keyword.replace("\\", "\\\\").replace('"', '\\"')
    if re.search(r"\s", keyword) or any(ch in keyword for ch in ["-", "/", "'", ","]):
        return f'"{keyword}"'
    return keyword


def with_extra_filters(query: str) -> str:
    filters = " ".join(part.strip() for part in EXTRA_QUERY_FILTERS if part.strip())
    return f"({query}) {filters}" if filters else query


def batch_keywords_for_queries(keywords: list[str], char_limit: int = QUERY_CHAR_LIMIT) -> list[str]:
    batches: list[str] = []
    current_terms: list[str] = []

    def render(terms: list[str]) -> str:
        return with_extra_filters(" OR ".join(terms))

    for keyword in keywords:
        term = quote_query_term(keyword)
        if len(with_extra_filters(term)) > char_limit:
            print(f"Skipping overlong keyword for query limit: {keyword[:80]}")
            continue
        candidate_terms = [*current_terms, term]
        if current_terms and len(render(candidate_terms)) > char_limit:
            batches.append(render(current_terms))
            current_terms = [term]
        else:
            current_terms = candidate_terms

    if current_terms:
        batches.append(render(current_terms))
    return batches


keywords = load_keywords_from_files(KEYWORD_FILES)
query_batches = batch_keywords_for_queries(keywords)

print(f"Keyword files : {len(KEYWORD_FILES)}")
for path in KEYWORD_FILES:
    print(f"- {path.relative_to(ROOT_DIR)}")
print(f"Keywords      : {len(keywords)}")
print(f"Query batches : {len(query_batches)}")
print("First query:")
print(query_batches[0] if query_batches else "<none>")

Keyword files : 5
- docs/keywords/by_language/cebuano_keywords.csv
- docs/keywords/by_language/english/english_keywords.csv
- docs/keywords/by_language/filipino_keywords.csv
- docs/keywords/by_language/hiligaynon_keywords.csv
- docs/keywords/by_language/ilocano_keywords.csv
Keywords      : 68
Query batches : 2
First query:
hubak OR covid OR cough OR "dry cough" OR fever OR "high temperature" OR "sore throat" OR dyspnea OR "difficulty of breathing" OR headache OR myalgia OR "body pain" OR "body aches" OR "muscle pain" OR rashes OR diarrhea OR "loose bowel movement" OR tiredness OR fatigue OR weakness OR chills OR colds OR "runny nose" OR "stuffy nose" OR "chest pain" OR "loss of smell" OR smell OR "loss of taste" OR taste OR anorexia OR "loss of appetite" OR conjunctivitis OR "red eyes" OR nausea OR vomiting OR "productive cough" OR "cough with phlegm" OR "low grade fever" OR "shortness of breath" OR "nasal congestion" OR "cough for more than three weeks" OR tuberculosis OR tb OR hemopt

In [17]:
SEARCH_ALL_URL = "https://api.x.com/2/tweets/search/all"
PUBLIC_TWEET_FIELDS = ["created_at", "id", "text", "public_metrics"]


def comma(values: list[str]) -> str:
    return ",".join(values)


def search_params(query: str, next_token: str | None = None, profile: str = DEFAULT_FIELD_PROFILE) -> dict[str, str | int]:
    params: dict[str, str | int] = {
        "query": query,
        "max_results": MAX_RESULTS,
        "sort_order": SORT_ORDER,
        "tweet.fields": comma(PUBLIC_TWEET_FIELDS),
    }
    if START_TIME:
        params["start_time"] = START_TIME
    if END_TIME:
        params["end_time"] = END_TIME
    if next_token:
        params["pagination_token"] = next_token
    return params


def summarize_api_errors(errors: list[dict[str, Any]] | None, limit: int = 10) -> dict[str, Any]:
    errors = errors or []
    title_counts = Counter(str(error.get("title", "<missing>")) for error in errors)
    field_counts = Counter(str(error.get("field", "")) for error in errors if error.get("field"))
    first_errors = [
        {key: error.get(key) for key in ["title", "detail", "field", "type"] if error.get(key)}
        for error in errors[:3]
    ]
    return {
        "count": len(errors),
        "titles": dict(title_counts.most_common(limit)),
        "fields": dict(field_counts.most_common(limit)),
        "first_errors": first_errors,
    }


def get_existing_ids(csv_paths: list[Path]) -> set[str]:
    ids: set[str] = set()
    for path in csv_paths:
        if not path.exists() or path.stat().st_size == 0:
            continue
        try:
            df = pd.read_csv(path, usecols=["id"], dtype=str)
        except ValueError:
            continue
        ids.update(df["id"].dropna().astype(str).str.replace("tweet-", "", regex=False).str.strip())
    return {post_id for post_id in ids if post_id}


existing_post_ids = get_existing_ids(sorted(OUTPUT_DIR.glob("*.csv")))
print(f"Existing Twitter/X IDs found for dedupe: {len(existing_post_ids)}")

Existing Twitter/X IDs found for dedupe: 3486


In [18]:
PREPROCESSED_TWITTER_COLUMNS = [
    "created_at",
    "id",
    "text",
    "like_count",
]


def flatten_post(post: dict[str, Any]) -> dict[str, Any]:
    public_metrics = post.get("public_metrics") or {}

    return {
        "created_at": post.get("created_at"),
        "id": str(post.get("id", "")).strip(),
        "text": post.get("text"),
        "like_count": public_metrics.get("like_count"),
    }


def append_jsonl(path: Path, item: dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(item, ensure_ascii=False, sort_keys=True) + "\n")


def trimmed_response(payload: dict[str, Any]) -> dict[str, Any]:
    return {
        "data": [flatten_post(post) for post in payload.get("data") or []],
        "meta": payload.get("meta") or {},
    }


def append_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    if not rows:
        return
    df = pd.DataFrame(rows).reindex(columns=PREPROCESSED_TWITTER_COLUMNS)
    file_exists = path.exists() and path.stat().st_size > 0
    df.to_csv(path, mode="a", index=False, header=not file_exists, encoding="utf-8-sig")


In [19]:
def request_search_page(session: requests.Session, query: str, next_token: str | None, profile: str) -> tuple[dict[str, Any], str, int]:
    params = search_params(query=query, next_token=next_token, profile=profile)
    response = session.get(SEARCH_ALL_URL, params=params, timeout=60)
    request_count = 1

    if response.status_code == 429:
        reset_at = response.headers.get("x-rate-limit-reset")
        sleep_seconds = REQUEST_SLEEP_SECONDS
        if reset_at and reset_at.isdigit():
            sleep_seconds = max(int(reset_at) - int(time.time()) + 2, REQUEST_SLEEP_SECONDS)
        print(f"Rate limited. Sleeping {sleep_seconds:.0f} seconds before retry.")
        time.sleep(sleep_seconds)
        response = session.get(SEARCH_ALL_URL, params=params, timeout=60)
        request_count += 1

    if not response.ok:
        raise requests.HTTPError(f"HTTP {response.status_code}: {response.text[:1000]}", response=response)

    return response.json(), DEFAULT_FIELD_PROFILE, request_count


def scrape_query(session: requests.Session, query: str, query_index: int, seen_ids: set[str], manifest: dict[str, Any]) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    next_token: str | None = None
    profile = DEFAULT_FIELD_PROFILE

    for page_number in range(1, MAX_PAGES_PER_QUERY + 1):
        payload, profile, request_count = request_search_page(session, query=query, next_token=next_token, profile=profile)
        manifest["request_count"] += request_count
        manifest["field_profiles_used"].append(profile)
        manifest["final_field_profile_by_query"][str(query_index)] = profile

        api_errors = payload.get("errors") or []
        if api_errors:
            manifest["api_error_summaries"].append({
                "query_index": query_index,
                "page": page_number,
                "field_profile": profile,
                "summary": summarize_api_errors(api_errors),
            })

        append_jsonl(JSONL_OUTPUT_FILE, {
            "run_id": RUN_ID,
            "query_index": query_index,
            "page": page_number,
            "field_profile": profile,
            "query": query,
            "start_time": START_TIME,
            "end_time": END_TIME,
            "response": trimmed_response(payload),
        })

        page_rows = []
        data = payload.get("data") or []
        for post in data:
            post_id = str(post.get("id", "")).strip()
            if not post_id or post_id in seen_ids:
                manifest["duplicate_count"] += 1
                continue
            seen_ids.add(post_id)
            page_rows.append(flatten_post(post))

        append_csv(CSV_OUTPUT_FILE, page_rows)
        rows.extend(page_rows)

        meta = payload.get("meta") or {}
        result_count = meta.get("result_count", len(data))
        error_count = len(api_errors)
        print(
            f"Query {query_index}, page {page_number}: "
            f"result_count={result_count}, data_len={len(data)}, "
            f"error_count={error_count}, field_profile={profile}, new_rows={len(page_rows)}"
        )

        if not data and api_errors:
            print(f"API returned no data with errors: {summarize_api_errors(api_errors)}")

        next_token = meta.get("next_token")
        if not next_token:
            break
        time.sleep(REQUEST_SLEEP_SECONDS)

    return rows

## Run Scrape

For a dry run, keep `query_batches[:1]` and `MAX_PAGES_PER_QUERY = 1`. Increase both after checking the generated CSV and JSONL.

In [20]:
if not BEARER_TOKEN:
    raise RuntimeError("Set X_BEARER_TOKEN in your environment before running the scraper.")
if not query_batches:
    raise RuntimeError("No query batches were built from keyword files.")

manifest: dict[str, Any] = {
    "run_id": RUN_ID,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "endpoint": SEARCH_ALL_URL,
    "keyword_dir": str(KEYWORD_DIR.relative_to(ROOT_DIR)),
    "keyword_files": [str(path.relative_to(ROOT_DIR)) for path in KEYWORD_FILES],
    "keyword_count": len(keywords),
    "query_batches": query_batches,
    "query_batch_count": len(query_batches),
    "query_char_limit": QUERY_CHAR_LIMIT,
    "extra_query_filters": EXTRA_QUERY_FILTERS,
    "start_time": START_TIME,
    "end_time": END_TIME,
    "max_results": MAX_RESULTS,
    "max_pages_per_query": MAX_PAGES_PER_QUERY,
    "sort_order": SORT_ORDER,
    "default_field_profile": DEFAULT_FIELD_PROFILE,
    "csv_output_file": str(CSV_OUTPUT_FILE.relative_to(ROOT_DIR)),
    "jsonl_output_file": str(JSONL_OUTPUT_FILE.relative_to(ROOT_DIR)),
    "manifest_output_file": str(MANIFEST_OUTPUT_FILE.relative_to(ROOT_DIR)),
    "request_count": 0,
    "row_count": 0,
    "duplicate_count": 0,
    "field_profiles_used": [],
    "final_field_profile_by_query": {},
    "api_error_summaries": [],
    "errors": [],
}

headers = {"Authorization": f"Bearer {BEARER_TOKEN}", "User-Agent": "HealthPHPlus-XFullArchiveScraper/1.0"}
seen_ids = set(existing_post_ids)
all_rows: list[dict[str, Any]] = []

with requests.Session() as session:
    session.headers.update(headers)
    for query_index, query in enumerate(query_batches, start=1):
        try:
            all_rows.extend(scrape_query(session, query=query, query_index=query_index, seen_ids=seen_ids, manifest=manifest))
        except Exception as exc:
            error = {"query_index": query_index, "query": query, "error": repr(exc)}
            manifest["errors"].append(error)
            print(f"Error on query {query_index}: {exc}")
        time.sleep(REQUEST_SLEEP_SECONDS)

manifest["row_count"] = len(all_rows)
manifest["completed_at"] = datetime.now(timezone.utc).isoformat()
manifest["field_profiles_used"] = sorted(set(manifest["field_profiles_used"]))

MANIFEST_OUTPUT_FILE.write_text(json.dumps(manifest, indent=2, ensure_ascii=False, sort_keys=True), encoding="utf-8")

print(f"Rows written       : {manifest['row_count']}")
print(f"Duplicates skipped : {manifest['duplicate_count']}")
print(f"Requests made      : {manifest['request_count']}")
print(f"API error pages    : {len(manifest['api_error_summaries'])}")
print(f"CSV                : {CSV_OUTPUT_FILE}")
print(f"JSONL              : {JSONL_OUTPUT_FILE}")
print(f"Manifest           : {MANIFEST_OUTPUT_FILE}")

Query 1, page 1: result_count=100, data_len=100, error_count=0, field_profile=public, new_rows=100
Query 1, page 2: result_count=100, data_len=100, error_count=0, field_profile=public, new_rows=100
Query 1, page 3: result_count=100, data_len=100, error_count=0, field_profile=public, new_rows=100
Query 2, page 1: result_count=100, data_len=100, error_count=0, field_profile=public, new_rows=100
Query 2, page 2: result_count=100, data_len=100, error_count=0, field_profile=public, new_rows=100
Query 2, page 3: result_count=100, data_len=100, error_count=0, field_profile=public, new_rows=100
Rows written       : 600
Duplicates skipped : 0
Requests made      : 6
API error pages    : 0
CSV                : /Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/data/raw/twitter/x_full_archive_20260515T062632Z.csv
JSONL              : /Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/data/raw/twitter/x_full_archive_20260515T062632Z.jsonl
Manifest           : /Users/angelodelapaz/Documents/GitHub

## Output Checks

In [22]:
if CSV_OUTPUT_FILE.exists() and CSV_OUTPUT_FILE.stat().st_size > 0:
    preview_df = pd.read_csv(CSV_OUTPUT_FILE, dtype=str)
    print(preview_df.shape)
    display(preview_df.head())
else:
    print("No CSV rows were written for this run.")

if JSONL_OUTPUT_FILE.exists() and JSONL_OUTPUT_FILE.stat().st_size > 0:
    with JSONL_OUTPUT_FILE.open("r", encoding="utf-8") as f:
        first_page = json.loads(next(f))
    first_response = first_page.get("response", {})
    print(first_page.keys())
    print("meta:", first_response.get("meta", {}))
    print("has_data:", bool(first_response.get("data")), "data_len:", len(first_response.get("data") or []))
    if first_response.get("data"):
        print("data keys:", list(first_response["data"][0].keys()))
else:
    print("No JSONL pages were written for this run.")

(600, 4)


,created_at,id,text,like_count
0,2026-05-15T06:26:22.000Z,2055172923506217428,RT @BrockMagnuss: My hot delivery guy got to t...,0
1,2026-05-15T06:26:22.000Z,2055172923501986225,"@Chino_Chulo Que va! Cap impost ha baixat, ni ...",0
2,2026-05-15T06:26:22.000Z,2055172922185007230,RT @strangersenjoy: He couldn't help but taste...,0
3,2026-05-15T06:26:22.000Z,2055172921631420585,RT @moana_vibe: taste my anal,0
4,2026-05-15T06:26:22.000Z,2055172921459659139,RT @noorielova: You can just tell the owner ha...,0


dict_keys(['end_time', 'field_profile', 'page', 'query', 'query_index', 'response', 'run_id', 'start_time'])
meta: {'newest_id': '2055172923506217428', 'next_token': 'b26v89c19zqg8o3juenifnn9x5j1o9n1tjoayy5tviqgt', 'oldest_id': '2055172838533832969', 'result_count': 100}
has_data: True data_len: 100
data keys: ['created_at', 'id', 'like_count', 'text']
